# Personality to Music Recommender (Phase 1)

This notebook implements the first stage of the project:
1. Validate whether the Young People Survey dataset can support personality-to-music prediction.
2. Build Big Five (OCEAN) proxy features from survey responses.
3. Train a baseline multi-output model to predict music taste scores.
4. Generate top-K genre recommendations from predicted preference scores.

In [23]:
from pathlib import Path

import warnings

import numpy as np

import pandas as pd

from sklearn.impute import SimpleImputer

from sklearn.metrics import mean_absolute_error, mean_squared_error

from sklearn.model_selection import train_test_split

from sklearn.multioutput import MultiOutputRegressor

from sklearn.ensemble import RandomForestRegressor

from sklearn.linear_model import RidgeCV



warnings.filterwarnings("ignore")

SEED = 42

np.random.seed(SEED)



PROJECT_ROOT = Path.cwd()

DATA_PATH = PROJECT_ROOT / "dataset" / "responses.csv"

COLUMNS_PATH = PROJECT_ROOT / "dataset" / "columns.csv"



print(f"Project root: {PROJECT_ROOT}")

print(f"Responses file exists: {DATA_PATH.exists()}")

print(f"Columns file exists: {COLUMNS_PATH.exists()}")

Project root: /Users/sauravkarki/repos/music
Responses file exists: True
Columns file exists: True


In [24]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Expected dataset at {DATA_PATH}. Put responses.csv under dataset/."
    )

df = pd.read_csv(DATA_PATH)
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print("Columns:")
print(df.columns.tolist())

Rows: 1010, Columns: 150
Columns:
['Music', 'Slow songs or fast songs', 'Dance', 'Folk', 'Country', 'Classical music', 'Musical', 'Pop', 'Rock', 'Metal or Hardrock', 'Punk', 'Hiphop, Rap', 'Reggae, Ska', 'Swing, Jazz', 'Rock n roll', 'Alternative', 'Latino', 'Techno, Trance', 'Opera', 'Movies', 'Horror', 'Thriller', 'Comedy', 'Romantic', 'Sci-fi', 'War', 'Fantasy/Fairy tales', 'Animated', 'Documentary', 'Western', 'Action', 'History', 'Psychology', 'Politics', 'Mathematics', 'Physics', 'Internet', 'PC', 'Economy Management', 'Biology', 'Chemistry', 'Reading', 'Geography', 'Foreign languages', 'Medicine', 'Law', 'Cars', 'Art exhibitions', 'Religion', 'Countryside, outdoors', 'Dancing', 'Musical instruments', 'Writing', 'Passive sport', 'Active sport', 'Gardening', 'Celebrities', 'Shopping', 'Science and technology', 'Theatre', 'Fun with friends', 'Adrenaline sports', 'Pets', 'Flying', 'Storm', 'Darkness', 'Heights', 'Spiders', 'Snakes', 'Rats', 'Ageing', 'Dangerous dogs', 'Fear of publi

In [25]:
MUSIC_COLUMNS = [
    "Dance", "Folk", "Country", "Classical music", "Musical", "Pop", "Rock",
    "Metal or Hardrock", "Punk", "Hiphop, Rap", "Reggae, Ska", "Swing, Jazz",
    "Rock n roll", "Alternative", "Latino", "Techno, Trance", "Opera"
]

available_music_cols = [c for c in MUSIC_COLUMNS if c in df.columns]
missing_music_cols = [c for c in MUSIC_COLUMNS if c not in df.columns]

print(f"Music target columns found: {len(available_music_cols)} / {len(MUSIC_COLUMNS)}")
if missing_music_cols:
    print("Missing expected music columns:")
    print(missing_music_cols)

music_stats = (
    df[available_music_cols]
    .apply(pd.to_numeric, errors="coerce")
    .describe()
)
display(music_stats)

Music target columns found: 17 / 17


,Dance,Folk,Country,Classical music,Musical,Pop,Rock,Metal or Hardrock,Punk,"Hiphop, Rap","Reggae, Ska","Swing, Jazz",Rock n roll,Alternative,Latino,"Techno, Trance",Opera
count,1006.000000,1005.000000,1005.000000,1003.000000,1008.000000,1007.000000,1004.000000,1007.000000,1002.000000,1006.000000,1003.000000,1004.000000,1003.000000,1003.000000,1002.000000,1003.000000,1009.000000
mean,3.113320,2.288557,2.123383,2.956132,2.761905,3.471698,3.761952,2.361470,2.456088,2.910537,2.769691,2.759960,3.141575,2.828514,2.842315,2.338983,2.139742
std,1.170568,1.138916,1.076136,1.252570,1.260845,1.161400,1.184861,1.372995,1.301105,1.375677,1.214434,1.257936,1.237269,1.347173,1.327902,1.324099,1.184094
min,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,2.000000,1.000000,1.000000,2.000000,2.000000,3.000000,3.000000,1.000000,1.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,1.000000,1.000000
50%,3.000000,2.000000,2.000000,3.000000,3.000000,4.000000,4.000000,2.000000,2.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,2.000000,2.000000
75%,4.000000,3.000000,3.000000,4.000000,4.000000,4.000000,5.000000,3.000000,3.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,3.000000,3.000000
max,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000


In [26]:
def suitability_report(frame: pd.DataFrame, target_cols: list[str]) -> pd.DataFrame:
    total_rows = len(frame)
    report_rows = []

    for col in target_cols:
        s = pd.to_numeric(frame[col], errors="coerce")
        missing_pct = float(s.isna().mean() * 100)
        nunique = int(s.nunique(dropna=True))
        min_val = float(s.min()) if s.notna().any() else np.nan
        max_val = float(s.max()) if s.notna().any() else np.nan

        report_rows.append({
            "column": col,
            "missing_pct": round(missing_pct, 2),
            "unique_values": nunique,
            "min": min_val,
            "max": max_val,
            "valid_for_model": (missing_pct < 35) and (nunique >= 3),
        })

    report = pd.DataFrame(report_rows).sort_values("missing_pct")

    duplicate_rows = int(frame.duplicated().sum())
    print(f"Duplicate rows: {duplicate_rows} ({duplicate_rows / total_rows:.2%})")

    accept_ratio = report["valid_for_model"].mean() if not report.empty else 0.0
    if accept_ratio >= 0.8:
        verdict = "Accept"
    elif accept_ratio >= 0.6:
        verdict = "Accept-with-limitations"
    else:
        verdict = "Reject-and-replace"

    print(f"Dataset suitability verdict: {verdict}")
    return report

target_quality = suitability_report(df, available_music_cols)
display(target_quality)

Duplicate rows: 0 (0.00%)
Dataset suitability verdict: Accept


,column,missing_pct,unique_values,min,max,valid_for_model
16,Opera,0.10,5,1.0,5.0,True
4,Musical,0.20,5,1.0,5.0,True
7,Metal or Hardrock,0.30,5,1.0,5.0,True
5,Pop,0.30,5,1.0,5.0,True
9,"Hiphop, Rap",0.40,5,1.0,5.0,True
0,Dance,0.40,5,1.0,5.0,True
2,Country,0.50,5,1.0,5.0,True
1,Folk,0.50,5,1.0,5.0,True
6,Rock,0.59,5,1.0,5.0,True
11,"Swing, Jazz",0.59,5,1.0,5.0,True


In [27]:
# Big Five proxy mapping using columns available in the survey.
BIG5_MAP = {
    "O": ["Reading", "Foreign languages", "Art exhibitions", "Fantasy/Fairy tales", "Psychology"],
    "C": ["Prioritising workload", "Thinking ahead", "Reliability", "Keeping promises", "Writing notes"],
    "E": ["Socializing", "Public speaking", "Fun with friends", "Energy levels", "Dancing"],
    "A": ["Empathy", "Compassion to animals", "Giving", "Charity", "Children"],
    "N": ["Mood swings", "Self-criticism", "Life struggles", "Getting angry", "Fear of public speaking"],
}

def build_big5_features(frame: pd.DataFrame, mapping: dict[str, list[str]]) -> pd.DataFrame:
    out = pd.DataFrame(index=frame.index)
    used = {}

    for trait, cols in mapping.items():
        existing = [c for c in cols if c in frame.columns]
        used[trait] = existing
        if not existing:
            out[trait] = np.nan
            continue

        values = frame[existing].apply(pd.to_numeric, errors="coerce")
        out[trait] = values.mean(axis=1)

    out = out.apply(lambda col: (col - col.mean()) / (col.std() + 1e-8), axis=0)

    print("Big Five proxy columns used:")
    for k, v in used.items():
        print(f"  {k}: {len(v)} columns")

    return out

X_big5 = build_big5_features(df, BIG5_MAP)
display(X_big5.head())

Big Five proxy columns used:
  O: 5 columns
  C: 5 columns
  E: 5 columns
  A: 5 columns
  N: 5 columns


,O,C,E,A,N
0,0.652757,0.001649,1.292303,0.732902,-2.216165
1,0.147235,0.279283,-0.469611,-1.636863,0.667409
2,1.411040,1.112184,1.292303,1.325343,0.667409
3,0.652757,0.279283,-2.231525,-1.636863,2.109196
4,-0.358287,0.001649,0.235154,0.140460,-0.486020


In [28]:
y = df[available_music_cols].apply(pd.to_numeric, errors="coerce")

mask = X_big5.notna().all(axis=1)
X = X_big5.loc[mask].copy()
y = y.loc[mask].copy()

imp = SimpleImputer(strategy="median")
X_imp = pd.DataFrame(imp.fit_transform(X), columns=X.columns, index=X.index)
y_imp = pd.DataFrame(imp.fit_transform(y), columns=y.columns, index=y.index)

X_train, X_test, y_train, y_test = train_test_split(
    X_imp, y_imp, test_size=0.2, random_state=SEED
)

print(f"Training rows: {len(X_train)}, Test rows: {len(X_test)}")

Training rows: 808, Test rows: 202


In [29]:
rf_model = MultiOutputRegressor(
    RandomForestRegressor(
        n_estimators=250,
        max_depth=12,
        random_state=SEED,
        n_jobs=-1,
    )
)

ridge_model = MultiOutputRegressor(
    RidgeCV(alphas=np.array([0.01, 0.1, 1.0, 10.0, 100.0, 1000.0], dtype=float))
)

rf_model.fit(X_train, y_train)
ridge_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
ridge_pred = ridge_model.predict(X_test)

def eval_regression(y_true, y_pred, label):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    print(f"{label}: MAE={mae:.4f}, RMSE={rmse:.4f}")

eval_regression(y_test, rf_pred, "RandomForest MultiOutput")
eval_regression(y_test, ridge_pred, "RidgeCV MultiOutput")

# print("\nSelected alpha per target (RidgeCV):")
# target_cols = list(y_train.columns)
# for i, est in enumerate(ridge_model.estimators_):
#     alpha = getattr(est, "alpha_", None)
#     if alpha is None:
#         print(f"{target_cols[i]}: alpha=unavailable")
#     else:
#         print(f"{target_cols[i]}: alpha={float(alpha):.2f}")

RandomForest MultiOutput: MAE=1.0283, RMSE=1.2424
RidgeCV MultiOutput: MAE=0.9782, RMSE=1.1758


## How mood is used (Phase 2)

Mood is not an input to the Ridge genre model. Keep these signals separate:

- RidgeCV model predicts genre affinity from personality features.
- Mood maps directly to Spotify audio feature parameters (for example `target_valence`).

Pipeline:

- Personality features -> RidgeCV -> genre seeds
- Mood slider -> Spotify `target_valence`
- Spotify recommendations endpoint combines both to return tracks

## Next Step

Phase 2 can add Spotify integration using:
- Top-K predicted genres as seed genres.
- Mood and listening history to adjust target valence/energy/tempo.
- Fallback logic when confidence is low or profile data is sparse.